In [18]:
def mod_inv(a, m):
    try:
        return pow(a, -1, m)
    except ValueError:
        raise ValueError("Modular inverse does not exist")


def dsa_sign(p, q, g, private_key, h, k):
    """
    DSA Signing Process
    r = (g^k mod p) mod q
    s = k^-1 * (h + private_key * r) mod q
    """
    # calculate r
    r = pow(g, k, p) % q

    # calculate s
    k_inv = mod_inv(k, q)
    s = (k_inv * (h + private_key * r)) % q

    return r, s


def dsa_verify(p, q, g, public_key, h, r, s):
    """
    DSA Verification Process
    w = s^-1 mod q
    u1 = (h * w) mod q
    u2 = (r * w) mod q
    v = ((g^u1 * public_key^u2) mod p) mod q
    """
    # Basic range check
    if not (0 < r < q) or not (0 < s < q):
        return False, None

    # calculate w
    w = mod_inv(s, q)

    # calculate u1 and u2
    u1 = (h * w) % q
    u2 = (r * w) % q

    # calculate v
    v = (pow(g, u1, p) * pow(public_key, u2, p) % p) % q

    return v == r, v


q = 101
p = 7879
alpha = 170
a = 75
beta = 4567
h = 52
k = 49

print("DSA Signing:")
r, s = dsa_sign(p, q, alpha, a, h, k)
print(f"Message Hash (h): {h}")
print(f"Random value (k): {k}")
print(f"Signature (r, s): ({r}, {s})")

print("\nDSA Verification:")
is_valid, v_value = dsa_verify(p, q, alpha, beta, h, r, s)
print(f"Verification value (v): {v_value}")
print(f"v == r: {v_value == r}")
print(f"Signature Valid: {is_valid}")

DSA Signing:
Message Hash (h): 52
Random value (k): 49
Signature (r, s): (59, 79)

DSA Verification:
Verification value (v): 59
v == r: True
Signature Valid: True


In [19]:
# --- Global Curve Parameters ---
P = 127    # Modulus p
A = 1    # Coefficient a
B = 26   # Coefficient b
N = 131  # Order of the group n

def point_add(p, q):
    """Adds two points p and q on the elliptic curve."""
    if p is None: return q
    if q is None: return p

    x1, y1 = p
    x2, y2 = q

    if x1 == x2 and (y1 != y2 or y1 == 0):
        return None  # Point at infinity

    if x1 == x2:
        # Point doubling: slope = (3x^2 + a) / 2y
        slope = (3 * x1**2 + A) * pow(2 * y1, -1, P)
    else:
        # Point addition: slope = (y2 - y1) / (x2 - x1)
        slope = (y2 - y1) * pow(x2 - x1, -1, P)

    slope %= P
    x3 = (slope**2 - x1 - x2) % P
    y3 = (slope * (x1 - x3) - y1) % P

    return (x3, y3)

def scalar_mul(k, p):
    """Computes k * p using the double-and-add algorithm."""
    result = None
    addend = p
    while k:
        if k & 1:
            result = point_add(result, addend)
        addend = point_add(addend, addend)
        k >>= 1
    return result

def ecdsa_sign(a, private_key, h, k):
    """Computes the ECDSA signature (r, s)."""
    # 1. R = k * a
    R = scalar_mul(k, a)
    r = R[0] % N
    
    # 2. s = k^-1 * (h + private_key * r) mod n
    s = (pow(k, -1, N) * (h + private_key * r)) % N
    return r, s

def ecdsa_verify(a, b, h, r, s):
    """Verifies the ECDSA signature."""
    # 1. w = s^-1 mod n
    w = pow(s, -1, N)
    
    # 2. u1 = h*w, u2 = r*w
    u1 = (h * w) % N
    u2 = (r * w) % N
    
    # 3. p = u1*a + u2*b
    p = point_add(scalar_mul(u1, a), scalar_mul(u2, b))
    
    if p is None: return False, None
    return (p[0] % N == r), p

a = (2, 6)      # Base point
m = 54          # Private key
h = 10          # Message hash
k = 75          # Random nonce

# (a) Compute Public Key b = m * a
b = scalar_mul(m, a)
print(f"(a) Public Key b: {b}")

# (b) Compute Signature (r, s)
r, s = ecdsa_sign(a, m, h, k)
print(f"(b) Signature (r, s): ({r}, {s})")

# (c) Verify Signature
is_valid, v_point = ecdsa_verify(a, b, h, r, s)
print(f"(c) Verification Point p: {v_point}")
print(f"    x_P mod n == r: {v_point[0] % N == r if v_point else False}")
print(f"    Signature Valid: {is_valid}")

(a) Public Key b: (24, 44)
(b) Signature (r, s): (88, 60)
(c) Verification Point p: (88, 55)
    x_P mod n == r: True
    Signature Valid: True


In [20]:
def recover_secret_key(h, r, q):
    """
    Adversary's function to recover the secret key 'a' given a signature where s = 0.
    """
    # a = (-h * r^-1) mod q
    r_inv = pow(r, -1, q)
    a_recovered = (-h * r_inv) % q
    return a_recovered

q = 101
p = 7879
g = 170
a_secret = 75  # The secret key we want to steal
k = 49         # The random nonce

# Calculate r as usual
r = pow(g, k, p) % q

# To simulate the vulnerability, we force s = 0.
# Since s = k^-1 * (h + a*r) mod q, s is 0 if (h + a*r) is a multiple of q.
# Calculate the specific hash 'h' that would cause s to be 0.
h_forced = (-a_secret * r) % q

print(f"Environment:")
print(f"Public q: {q}")
print(f"Public g: {g}")
print(f"Secret key a: {a_secret} (Unknown to adversary)")
print(f"Random nonce k: {k} (Unknown to adversary)")

print(f"\n'Broken' Signature:")
print(f"Message Hash h: {h_forced}")
print(f"Signature r: {r}")
print(f"Signature s: 0")

print(f"\nAdversary Attack:")
# The adversary only knows h, r, and q
recovered_a = recover_secret_key(h_forced, r, q)
print(f"Adversary computes a = (-h * r^-1) mod q")
print(f"Recovered Secret Key: {recovered_a}")

if recovered_a == a_secret:
    print(f"\nrecovered_a == a_secret: {recovered_a == a_secret}, Secret key recovered")

Environment:
Public q: 101
Public g: 170
Secret key a: 75 (Unknown to adversary)
Random nonce k: 49 (Unknown to adversary)

'Broken' Signature:
Message Hash h: 19
Signature r: 59
Signature s: 0

Adversary Attack:
Adversary computes a = (-h * r^-1) mod q
Recovered Secret Key: 75

recovered_a == a_secret: True, Secret key recovered


In [21]:
q = 101
p = 7879
g = 170
a_secret = 75
y_public = pow(g, a_secret, p)

# Find a k that results in r = 0
# In a real attack, the adversary just sees a signature where r=0.
# Here, we find such a k to simulate the scenario.
k_special = None
for k_test in range(1, q):
    if (pow(g, k_test, p) % q) == 0:
        k_special = k_test
        break

# Alice signs message h1 using k_special
h1 = 52
r1 = pow(g, k_special, p) % q  # This is 0
s1 = (pow(k_special, -1, q) * (h1 + a_secret * r1)) % q

print(f"Original Signature (r=0):")
print(f"Message h1: {h1}")
print(f"Signature: (r={r1}, s={s1})")

# Adversary recovers k
# h1, r1=0, s1, and q are known
k_recovered = (h1 * pow(s1, -1, q)) % q
print(f"\nRecovering k:")
print(f"Adversary computes k = (h1 * s1^-1) mod q")
print(f"Recovered k: {k_recovered}")

# Selective Forgery
# sign a new message h2
h2 = 420  # arbitrary hash
r_forge = 0
# set u1 = k_recovered, solving for s_forge:
# k = h2 * s_forge^-1  => s_forge = h2 * k^-1
s_forge = (h2 * pow(k_recovered, -1, q)) % q

print(f"\nSelective Forgery:")
print(f"Target Message h2: {h2}")
print(f"Forged Signature: (r={r_forge}, s={s_forge})")

# Verification
is_valid = dsa_verify(p, q, g, y_public, h2, r_forge, s_forge)
if is_valid:
    print(f"\nforged signature is valid")

Original Signature (r=0):
Message h1: 52
Signature: (r=0, s=81)

Recovering k:
Adversary computes k = (h1 * s1^-1) mod q
Recovered k: 58

Selective Forgery:
Target Message h2: 420
Forged Signature: (r=0, s=56)

forged signature is valid
